# Career Path Advisor Chatbot

## 📌 Project Goal
The goal of this project is to build an **advisor chatbot** that helps individuals make informed career decisions. The chatbot will recommend possible **career paths** based on the skills a person currently possesses and also suggest **additional skills** they can learn to expand their opportunities and transition into different fields.

## 🔍 Problem Statement
Many people have valuable skills but struggle to identify:
- The types of jobs or roles they are best suited for.
- What complementary skills they need to acquire in order to pivot into new industries or advance in their careers.

This lack of clarity often leads to underemployment or missed opportunities.

## 💡 Solution
The Career Path Advisor Chatbot will:
1. Accept a list of **current skills** from a user.
2. Map these skills to **job descriptions** in a dataset of careers.
3. Recommend relevant **job titles and career paths** that align with the user’s existing skill set.
4. Suggest **new skills to learn** that complement the current ones and enable the user to transition into other fields or improve employability.

## 📊 Dataset
The project uses a dataset of **job descriptions**, including:
- **Job Titles**
- **Required Skills**
- **Job Descriptions**

This dataset provides the foundation for matching user skills to career opportunities.

## 🔧 Methodology
- **Data Preprocessing**: Clean and structure job descriptions and skill requirements.
- **Skill Extraction & Matching**: Use NLP techniques to extract skills from text and match them to user input.
- **Recommendation System**: Suggest suitable jobs and additional skills for career advancement.
- **Chatbot Interface**: Deploy the solution as an interactive chatbot that users can engage with naturally.

## 🚀 Expected Outcome
By the end of this project, users will have a **personalized career advisor chatbot** that:
- Guides them toward roles that match their current skills.
- Provides clear advice on **what to learn next** to remain competitive and versatile in the job market.

This tool can empower individuals to take control of their career growth and explore opportunities they may not have considered otherwise.


# THE FIRST STEP START HERE.

The first step is to call up the dataset that we got from huggingface and was saved to our local computer.

In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import random
import contractions
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
import datetime
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings("ignore")

[nltk_data] Error loading stopwords: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
[nltk_data] Error loading wordnet: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>


In [3]:
# import pandas so that it can be used to read the dataset we had got from huggingface and display the first five rows.
df = pd.read_csv("Joblisting.csv")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'Joblisting.csv'

## Data Preprocessing.

In [7]:
#check if there are any duplicates in the dataset.
df.duplicated().sum()

0

In [8]:
# check for missing values
df.isna().sum()

Job Id                 0
Experience             0
Qualifications         0
Salary Range           0
location               0
Country                0
latitude               0
longitude              0
Work Type              0
Company Size           0
Job Posting Date       0
Preference             0
Contact Person         0
Contact                0
Job Title              0
Role                   0
Job Portal             0
Job Description        0
Benefits               0
skills                 0
Responsibilities       0
Company                0
Company Profile     5478
dtype: int64

In [9]:
df1 = df[["Job Title","skills","Experience"]]
df1

,Job Title,skills,Experience
0,Digital Marketing Specialist,"Social media platforms (e.g., Facebook, Twitte...",5 to 15 Years
1,Web Developer,"HTML, CSS, JavaScript Frontend frameworks (e.g...",2 to 12 Years
2,Operations Manager,Quality control processes and methodologies St...,0 to 12 Years
3,Network Engineer,Wireless network design and architecture Wi-Fi...,4 to 11 Years
4,Event Manager,Event planning Conference logistics Budget man...,1 to 12 Years
...,...,...,...
1615935,Mechanical Engineer,"Mechanical engineering CAD software (e.g., Sol...",0 to 12 Years
1615936,IT Manager,Strategic IT planning Leadership and managemen...,2 to 14 Years
1615937,Mechanical Engineer,"Mechanical engineering CAD software (e.g., Sol...",4 to 15 Years
1615938,HR Coordinator,Training program coordination Training materia...,5 to 15 Years


In [10]:
df1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1615940 entries, 0 to 1615939
Data columns (total 3 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   Job Title   1615940 non-null  object
 1   skills      1615940 non-null  object
 2   Experience  1615940 non-null  object
dtypes: object(3)
memory usage: 37.0+ MB


In [11]:
df1.rename(columns={
    'Job Title': 'Job_Title',
    'skills': 'Skills',
    'Experience': 'Experience'
}, inplace=True)
df1.head()

,Job_Title,Skills,Experience
0,Digital Marketing Specialist,"Social media platforms (e.g., Facebook, Twitte...",5 to 15 Years
1,Web Developer,"HTML, CSS, JavaScript Frontend frameworks (e.g...",2 to 12 Years
2,Operations Manager,Quality control processes and methodologies St...,0 to 12 Years
3,Network Engineer,Wireless network design and architecture Wi-Fi...,4 to 11 Years
4,Event Manager,Event planning Conference logistics Budget man...,1 to 12 Years


In [12]:
# Inspecting the unque classes of Job Title in the dataset for possible cleaning and normalization
df1.Job_Title.unique()

array(['Digital Marketing Specialist', 'Web Developer',
       'Operations Manager', 'Network Engineer', 'Event Manager',
       'Software Tester', 'Teacher', 'UX/UI Designer', 'Wedding Planner',
       'QA Analyst', 'Litigation Attorney', 'Mechanical Engineer',
       'Network Administrator', 'Account Manager', 'Brand Manager',
       'Social Worker', 'Social Media Coordinator',
       'Email Marketing Specialist', 'HR Generalist', 'Legal Assistant',
       'Nurse Practitioner', 'Account Director', 'Software Engineer',
       'Purchasing Agent', 'Sales Consultant', 'Civil Engineer',
       'Network Security Specialist', 'UI Developer', 'Financial Planner',
       'Event Planner', 'Psychologist', 'Electrical Designer',
       'Data Analyst', 'Technical Writer', 'Tax Consultant',
       'Account Executive', 'Systems Administrator',
       'Database Administrator', 'Research Analyst', 'Data Entry Clerk',
       'Registered Nurse', 'Investment Analyst', 'Speech Therapist',
       'Sales M

In [13]:
# Inspecting the unque classes of Skills in the dataset for possible cleaning and normalization
df1.Skills.unique()

array(['Social media platforms (e.g., Facebook, Twitter, Instagram) Content creation and scheduling Social media analytics and insights Community engagement Paid social advertising',
       'HTML, CSS, JavaScript Frontend frameworks (e.g., React, Angular) User experience (UX)',
       'Quality control processes and methodologies Statistical process control (SPC) Root cause analysis and corrective action Quality management systems (e.g., ISO 9001) Compliance and regulatory knowledge',
       'Wireless network design and architecture Wi-Fi standards and protocols RF (Radio Frequency) planning and optimization Wireless security protocols Troubleshooting wireless network issues',
       'Event planning Conference logistics Budget management Vendor coordination Marketing and promotion Client relations',
       'Quality assurance processes Testing methodologies (e.g., manual, automated) Bug tracking and reporting Test case development Regression testing',
       'Teaching pedagogy Classroom 

In [14]:
# Inspecting the unque classes of experience in the dataset for possible cleaning and normalization
df1.Experience.unique()

array(['5 to 15 Years', '2 to 12 Years', '0 to 12 Years', '4 to 11 Years',
       '1 to 12 Years', '4 to 12 Years', '3 to 15 Years', '2 to 8 Years',
       '2 to 9 Years', '1 to 10 Years', '3 to 10 Years', '1 to 8 Years',
       '1 to 9 Years', '5 to 14 Years', '0 to 11 Years', '3 to 12 Years',
       '5 to 9 Years', '0 to 15 Years', '0 to 10 Years', '2 to 14 Years',
       '3 to 9 Years', '4 to 15 Years', '2 to 10 Years', '4 to 8 Years',
       '3 to 8 Years', '1 to 14 Years', '1 to 13 Years', '0 to 8 Years',
       '5 to 10 Years', '2 to 13 Years', '4 to 9 Years', '1 to 15 Years',
       '4 to 10 Years', '5 to 12 Years', '0 to 13 Years', '4 to 14 Years',
       '1 to 11 Years', '4 to 13 Years', '0 to 9 Years', '5 to 8 Years',
       '2 to 15 Years', '5 to 13 Years', '5 to 11 Years', '0 to 14 Years',
       '3 to 13 Years', '2 to 11 Years', '3 to 11 Years', '3 to 14 Years'],
      dtype=object)

In [15]:
#Cleaning & Normalizing Columns
def clean_column(text):
    text = str(text).lower()  # convert text to lowercase
    text = re.sub(r'[^a-z\s/]', '', text)  # remove punctuation except slashes
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    text = re.sub(r'\(.*?\)', '', text)  # remove parentheses and contents
    text = re.sub(r'[^a-z,\s]', '', text)  # remove special characters except commas
    return text

df1['Job_Title'] = df1['Job_Title'].apply(clean_column)
df1['Skills'] = df1['Skills'].apply(clean_column)

In [34]:
# dropping the experience column
df1.drop(columns=['Experience'], inplace=True)

In [36]:
df1.head()

,Job_Title,Skills,Min_Experience,Max_experience
0,digital marketing specialist,social media platforms eg facebook twitter ins...,5,15
1,web developer,html css javascript frontend frameworks eg rea...,2,12
2,operations manager,quality control processes and methodologies st...,0,12
3,network engineer,wireless network design and architecture wifi ...,4,11
4,event manager,event planning conference logistics budget man...,1,12


In [38]:
# Text preprocessing (stopwords, lemmatization)
def text_preprocessing(text):
    # Convert to lowercase
    text = str(text).lower()

    # Tokenize text into individual words
    tokens = word_tokenize(text)

    # Initialize lemmatizer
    lemmatizer = WordNetLemmatizer()

    # Load English stopwords
    stop_words = set(stopwords.words('english'))

    # Lemmatize each token (reduce word to base/dictionary form)
    # Skip words that are stopwords
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens if token not in stop_words]

    # Join the cleaned tokens back into a single string
    clean_text = " ".join(lemmatized_tokens)

    # Return final cleaned text
    return clean_text

In [40]:
df1['Job_Title_Clean'] = df1['Job_Title'].apply(text_preprocessing)
df1['Skills_Clean'] = df1['Skills'].apply(text_preprocessing)

In [44]:
# dropping the min/maxexperience column
df1.drop(columns=['Min_Experience', 'Max_experience'], inplace=True)

In [46]:
df1.head()

,Job_Title,Skills,Job_Title_Clean,Skills_Clean
0,digital marketing specialist,social media platforms eg facebook twitter ins...,digital marketing specialist,social medium platform eg facebook twitter ins...
1,web developer,html css javascript frontend frameworks eg rea...,web developer,html cs javascript frontend framework eg react...
2,operations manager,quality control processes and methodologies st...,operation manager,quality control process methodology statistica...
3,network engineer,wireless network design and architecture wifi ...,network engineer,wireless network design architecture wifi stan...
4,event manager,event planning conference logistics budget man...,event manager,event planning conference logistics budget man...


In [48]:
# Save the cleaned DataFrame to a CSV file without the index column
df1.to_csv('Joblisting_2.csv', index=False)

In [56]:
df2 = pd.read_csv('Joblisting_2.csv')
df2.head()

,Job_Title,Skills,Job_Title_Clean,Skills_Clean
0,digital marketing specialist,social media platforms eg facebook twitter ins...,digital marketing specialist,social medium platform eg facebook twitter ins...
1,web developer,html css javascript frontend frameworks eg rea...,web developer,html cs javascript frontend framework eg react...
2,operations manager,quality control processes and methodologies st...,operation manager,quality control process methodology statistica...
3,network engineer,wireless network design and architecture wifi ...,network engineer,wireless network design architecture wifi stan...
4,event manager,event planning conference logistics budget man...,event manager,event planning conference logistics budget man...


In [58]:
# Creating word count columns for job title and skills
df2['Job_Title_Word_Count'], df2['Skills_Word_Count'] = (
    df2['Job_Title_Clean'].apply(lambda x: len(str(x).split())),
    df2['Skills_Clean'].apply(lambda x: len(str(x).split()))
)

In [60]:
df2.head()

,Job_Title,Skills,Job_Title_Clean,Skills_Clean,Job_Title_Word_Count,Skills_Word_Count
0,digital marketing specialist,social media platforms eg facebook twitter ins...,digital marketing specialist,social medium platform eg facebook twitter ins...,3,19
1,web developer,html css javascript frontend frameworks eg rea...,web developer,html cs javascript frontend framework eg react...,2,11
2,operations manager,quality control processes and methodologies st...,operation manager,quality control process methodology statistica...,2,21
3,network engineer,wireless network design and architecture wifi ...,network engineer,wireless network design architecture wifi stan...,2,19
4,event manager,event planning conference logistics budget man...,event manager,event planning conference logistics budget man...,2,12


In [135]:
# Initializing the vectorizers
job_title_vectorizer = TfidfVectorizer(stop_words='english')
skills_vectorizer = TfidfVectorizer(stop_words='english')

# Fit and transform dataset text
job_title_tfidf = job_title_vectorizer.fit_transform(df2['Job_Title_Clean'])
skills_tfidf = skills_vectorizer.fit_transform(df2['Skills_Clean'])

# Example for user input (convert to same TF-IDF space)
user_job_input = input("Enter desired job title: ")   # example user query
user_skills_input = input("Enter your skills: ")  # example user query

user_job_tfidf = job_title_vectorizer.transform([user_job_input])
user_skills_tfidf = skills_vectorizer.transform([user_skills_input])

Enter desired job title:  Banking
Enter your skills:  accounting, book-keeping


In [137]:
from sklearn.metrics.pairwise import cosine_similarity

In [139]:
# Job title similarity
job_similarities = cosine_similarity(user_job_tfidf, job_title_tfidf).flatten()

# Skills similarity
skills_similarities = cosine_similarity(user_skills_tfidf, skills_tfidf).flatten()

# Combine both similarities (weighted average, you can adjust weights)
combined_similarity = (0.5 * job_similarities) + (0.5 * skills_similarities)


In [147]:
# Get top 3 job matches
top_indices = combined_similarity.argsort()[::-1][:3]

results = df2.iloc[top_indices][['Job_Title_Clean', 'Skills_Clean']]
results['Similarity_Score'] = combined_similarity[top_indices]


# Overlapping Skills
# Convert user skills to a set of tokens
user_skills_set = set(user_skills_input.lower().replace(",", " ").split())

# Function to extract overlapping skills
def get_overlap(job_skills):
    job_skills_set = set(job_skills.lower().split())
    return list(user_skills_set & job_skills_set)  # intersection

# Apply overlap detection
results['Overlapping_Skills'] = results['Skills_Clean'].apply(get_overlap)

# Final display
results = results[['Job_Title_Clean', 'Skills_Clean', 'Overlapping_Skills', 'Similarity_Score']]
results


,Job_Title_Clean,Skills_Clean,Overlapping_Skills,Similarity_Score
209651,financial controller,accounting financial reporting financial audit...,[accounting],0.211225
1163888,financial controller,accounting financial reporting financial audit...,[accounting],0.211225
488446,financial controller,accounting financial reporting financial audit...,[accounting],0.211225


In [159]:
# Career Advice Section

# Function to extract missing skills
def get_missing(job_skills):
    job_skills_set = set(job_skills.lower().split())
    return list(job_skills_set - user_skills_set)

# Generate career advice for the top job matches
career_advice = []
for i in top_indices:
    job_title = df2.loc[i, 'Job_Title_Clean']
    job_skills = df2.loc[i, 'Skills_Clean']
    
    overlapping = get_overlap(job_skills)
    missing = get_missing(job_skills)
    
    advice = {
        "Job Title": job_title,
        "Required Skills": job_skills,
        "Your Matching Skills": overlapping,
        "Suggested Skills to Learn": missing,
        "Career Advice":f"Consider learning skills in {missing} to increase your chances."
    }
    career_advice.append(advice)

career_advice_df_2 = pd.DataFrame(career_advice)
career_advice_df_2


,Job Title,Required Skills,Your Matching Skills,Suggested Skills to Learn,Career Advice
0,financial controller,accounting financial reporting financial audit...,[accounting],"[gaap, audit, accepted, reporting, principle, ...","Consider learning skills in ['gaap', 'audit', ..."
1,financial controller,accounting financial reporting financial audit...,[accounting],"[gaap, audit, accepted, reporting, principle, ...","Consider learning skills in ['gaap', 'audit', ..."
2,financial controller,accounting financial reporting financial audit...,[accounting],"[gaap, audit, accepted, reporting, principle, ...","Consider learning skills in ['gaap', 'audit', ..."
